# 利用rating2inter.ipynb中U/I的index对features进行一一对应(meta-text)
- Reindex item feature ID with IDs generated in 0rating2inter.ipynb

In [1]:
import os
import pandas as pd

In [2]:
os.chdir('../data/grocery')
os.getcwd()

'/home/hun/MMRec_AlignRec/data/grocery'

In [3]:
# load item mapping
i_id_mapping = 'i_id_mapping.csv'
df = pd.read_csv(i_id_mapping, sep='\t')
print(f'shape: {df.shape}')
df[:4]

shape: (8713, 2)


,asin,itemID
0,616719923X,0
1,9742356831,1
2,B00004S1C5,2
3,B0000531B7,3


In [4]:

import gzip, json
meta_file = 'meta_Grocery_and_Gourmet_Food.json.gz'

print('0 Extracting U-I interactions.')

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

meta_df = getDF(meta_file)

print(f'Total records: {meta_df.shape}')
meta_df[:3]

0 Extracting U-I interactions.
Total records: (171760, 9)


,asin,description,title,imUrl,related,salesRank,categories,price,brand
0,0657745316,This is real vanilla extract made with only 3 ...,100 Percent All Natural Vanilla Extract,http://ecx.images-amazon.com/images/I/41gFi5h0...,{'also_viewed': ['B001GE8N4Y']},{'Grocery & Gourmet Food': 374004},[[Grocery & Gourmet Food]],NaN,NaN
1,0700026444,"Silverpot Tea, Pure Darjeeling, is an exquisit...",Pure Darjeeling Tea: Loose Leaf,http://ecx.images-amazon.com/images/I/51hs8sox...,NaN,{'Grocery & Gourmet Food': 620307},[[Grocery & Gourmet Food]],NaN,NaN
2,1403796890,Must have for any WWE Fan\n \n \n \nFeaturing ...,WWE Kids Todler Velvet Slippers featuring John...,http://ecx.images-amazon.com/images/I/518SEST5...,NaN,NaN,[[Grocery & Gourmet Food]],3.99,NaN


In [5]:
# remapping
map_dict = dict(zip(df['asin'], df['itemID']))

meta_df['itemID'] = meta_df['asin'].map(map_dict)
meta_df.dropna(subset=['itemID'], inplace=True)
meta_df['itemID'] = meta_df['itemID'].astype('int64')
#meta_df['description'] = meta_df['description'].fillna(" ")
meta_df.sort_values(by=['itemID'], inplace=True)

print(f'shape: {meta_df.shape}')
meta_df[:2]

shape: (8713, 10)


,asin,description,title,imUrl,related,salesRank,categories,price,brand,itemID
17,616719923X,Green Tea Flavor Kit Kat have quickly become t...,Japanese Kit Kat Maccha Green Tea Flavor (5 Ba...,http://ecx.images-amazon.com/images/I/51LdEao6...,"{'also_bought': ['B00FD63L5W', 'B0047YG5UY', '...",{'Grocery & Gourmet Food': 37305},[[Grocery & Gourmet Food]],NaN,NaN,0
31,9742356831,Used to make various curry soups and stir fry ...,Mae Ploy Thai Green Curry Paste - 14 oz jar,http://ecx.images-amazon.com/images/I/41pQp67A...,"{'also_bought': ['B000EI2LLO', 'B000EICISA', '...",{'Grocery & Gourmet Food': 3434},[[Grocery & Gourmet Food]],7.23,Mae Ploy,1


In [6]:
ori_cols = meta_df.columns.tolist()

ret_cols = [ori_cols[-1]] + ori_cols[:-1]
print(f'new column names: {ret_cols}')

new column names: ['itemID', 'asin', 'description', 'title', 'imUrl', 'related', 'salesRank', 'categories', 'price', 'brand']


In [7]:
meta_df[:3]

,asin,description,title,imUrl,related,salesRank,categories,price,brand,itemID
17,616719923X,Green Tea Flavor Kit Kat have quickly become t...,Japanese Kit Kat Maccha Green Tea Flavor (5 Ba...,http://ecx.images-amazon.com/images/I/51LdEao6...,"{'also_bought': ['B00FD63L5W', 'B0047YG5UY', '...",{'Grocery & Gourmet Food': 37305},[[Grocery & Gourmet Food]],NaN,NaN,0
31,9742356831,Used to make various curry soups and stir fry ...,Mae Ploy Thai Green Curry Paste - 14 oz jar,http://ecx.images-amazon.com/images/I/41pQp67A...,"{'also_bought': ['B000EI2LLO', 'B000EICISA', '...",{'Grocery & Gourmet Food': 3434},[[Grocery & Gourmet Food]],7.23,Mae Ploy,1
37,B00004S1C5,"From Easter eggs to colorful cookies, Spectrum...","Ateco Food Coloring Kit, 6 colors",http://ecx.images-amazon.com/images/I/41F75K9F...,"{'also_bought': ['B0000CFMLT', 'B002PO3KBK', '...",{'Kitchen & Dining': 4494},"[[Grocery & Gourmet Food, Cooking & Baking, Fo...",9.76,HIC Harold Import Co.,2


In [8]:
ret_df = meta_df[ret_cols]
# dump
ret_df.to_csv(os.path.join('./', 'meta-grocery.csv'), index=False)
print('done!')

done!


## Reload

In [ ]:
indexed_df = pd.read_csv('meta-grocery.csv')
print(f'shape: {indexed_df.shape}')
indexed_df[:4]

shape: (10672, 10)


,itemID,asin,description,price,imUrl,related,salesRank,categories,title,brand
0,0,0700099867,Dirt 3 is a popular rally racing game for Play...,246.63,http://ecx.images-amazon.com/images/I/41xSe3Sp...,"{'also_bought': ['B00488PZ0U', 'B008BT80SQ', '...",{'Video Games': 6629},"[['Video Games', 'PC', 'Games']]",NaN,NaN
1,1,6050036071,These are the official Disney Microphones for ...,39.99,http://ecx.images-amazon.com/images/I/31Sr03gb...,"{'also_bought': ['B002197J3O', 'B0028ZJ4ZW', '...",{'Video Games': 1673},"[['Video Games', 'PlayStation 3', 'Accessories...",NaN,NaN
2,2,7100027950,Nintendo's thematic action-adventure sequel to...,105.63,http://ecx.images-amazon.com/images/I/41Sf7in3...,"{'also_bought': ['B004CHLNWQ', 'B00012D0SG', '...",{'Video Games': 12852},"[['Video Games', 'Kids & Family'], ['Video Gam...",NaN,NaN
3,3,7293000936,Having stunning Nintendo Wii High Definition G...,6.05,http://ecx.images-amazon.com/images/I/41FskDVP...,"{'also_bought': ['B004WLRQAU', 'B0045FCKVI', '...",{'Video Games': 16102},"[['Video Games', 'Wii', 'Accessories', 'Cables...",NaN,NaN


In [10]:
## Reload

i_uni = indexed_df['itemID'].unique()

print(f'# of unique items: {len(i_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(i_uni), max(i_uni)))

# of unique items: 10672
min/max of unique learners: 0/10671
